<a href="https://colab.research.google.com/github/inspire2029-sudo/phantoms-ai-week2/blob/main/W2D4_Automation_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
!pip install -q requests schedule

In [44]:
import requests
import sqlite3
import json
import schedule
import time
from datetime import datetime

In [45]:
import requests

API_KEY = "73c46057cc9e21175d81e5c5811f319f"

API_URL = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Alexandria,EG",
    "appid": API_KEY,
    "units": "metric"
}

response = requests.get(API_URL, params=params, timeout=10)

print("Status Code:", response.status_code)

if response.status_code == 200:
    print("Authentication successful")
    data = response.json()
    print(data)
else:
    print("Request failed")
    print(response.text)

Status Code: 200
Authentication successful
{'coord': {'lon': 29.9553, 'lat': 31.2156}, 'weather': [{'id': 802, 'main': 'Clouds', 'description': 'scattered clouds', 'icon': '03n'}], 'base': 'stations', 'main': {'temp': 25.98, 'feels_like': 25.98, 'temp_min': 25.98, 'temp_max': 25.98, 'pressure': 1017, 'humidity': 73, 'sea_level': 1017, 'grnd_level': 1017}, 'visibility': 10000, 'wind': {'speed': 6.17, 'deg': 350}, 'clouds': {'all': 32}, 'dt': 1790268448, 'sys': {'type': 1, 'id': 2511, 'country': 'EG', 'sunrise': 1790221764, 'sunset': 1790265314}, 'timezone': 10800, 'id': 361058, 'name': 'Alexandria', 'cod': 200}


In [46]:
import sqlite3
import json
from datetime import datetime

connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS api_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        api_data TEXT NOT NULL,
        timestamp TEXT NOT NULL
    )
""")

connection.commit()
connection.close()

print("Database and table are ready")

Database and table are ready


### `store.db  في  API  هنخزن الـ `

In [47]:
connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute(
    """
    INSERT INTO api_logs (api_data, timestamp)
    VALUES (?, ?)
    """,
    (
        json.dumps(data, ensure_ascii=False),
        datetime.now().isoformat()
    )
)

connection.commit()
connection.close()

print("API data stored successfully")

API data stored successfully


### `بتأكد انها دخلت فعلا`

In [48]:
connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute("""
    SELECT id, api_data, timestamp
    FROM api_logs
    ORDER BY id DESC
    LIMIT 1
""")

row = cursor.fetchone()

connection.close()

print("Last stored record:")
print(row)

Last stored record:
(8, '{"coord": {"lon": 29.9553, "lat": 31.2156}, "weather": [{"id": 802, "main": "Clouds", "description": "scattered clouds", "icon": "03n"}], "base": "stations", "main": {"temp": 25.98, "feels_like": 25.98, "temp_min": 25.98, "temp_max": 25.98, "pressure": 1017, "humidity": 73, "sea_level": 1017, "grnd_level": 1017}, "visibility": 10000, "wind": {"speed": 6.17, "deg": 350}, "clouds": {"all": 32}, "dt": 1790268448, "sys": {"type": 1, "id": 2511, "country": "EG", "sunrise": 1790221764, "sunset": 1790265314}, "timezone": 10800, "id": 361058, "name": "Alexandria", "cod": 200}', '2026-09-24T16:47:40.917964')


In [49]:
WEBHOOK_URL = "https://kilowatt-yoga-sixties.ngrok-free.dev/webhook"

webhook_response = requests.post(
    WEBHOOK_URL,
    json=data,
    timeout=10
)

print("Webhook Status Code:", webhook_response.status_code)
print("Webhook Response:", webhook_response.text)

Webhook Status Code: 200
Webhook Response: {"status":"received"}



In [50]:
WEBHOOK_URL = "https://kilowatt-yoga-sixties.ngrok-free.dev/webhook"

webhook_response = requests.post(
    WEBHOOK_URL,
    json=data,
    timeout=10
)

print("Webhook Status Code:", webhook_response.status_code)
print("Webhook Response:", webhook_response.json())

Webhook Status Code: 200
Webhook Response: {'status': 'received'}


### `كدا الاوثانتيكيشن تمام + سحب البيانات + تخزين البيانات + ارساله لـ الويب هوك  `
### `وباقي الـ اوتوميشن`

### `run_pipeline() الـ `

In [51]:
def run_pipeline(cycle_number):
    print(f"\n--- بدء الدورة رقم {cycle_number} ---")

    # 1.API هنجيب الداتا من الـ
    response = requests.get(
        API_URL,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        print("فشل الاتصال بالـ API")
        print("Status Code:", response.status_code)
        return False

    data = response.json()
    print(" تم احضار الداتا من API")

    # 2. تخزين البيانات في Database
    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute(
        """
        INSERT INTO api_logs (api_data, timestamp)
        VALUES (?, ?)
        """,
        (
            json.dumps(data, ensure_ascii=False),
            datetime.now().isoformat()
        )
    )

    connection.commit()
    connection.close()

    print("تم تخزين البيانات في Database")

    # 3. إرسال نفس البيانات إلى Webhook
    webhook_response = requests.post(
        WEBHOOK_URL,
        json=data,
        timeout=10
    )

    if webhook_response.status_code != 200:
        print("فشل إرسال البيانات إلى Webhook")
        print("Status Code:", webhook_response.status_code)
        print("Response:", webhook_response.text)
        return False

    print("تم إرسال البيانات إلى Webhook")

    print(f"تم تنفيذ الدورة رقم {cycle_number} بنجاح")

    return True

In [52]:
run_pipeline(1)


--- بدء الدورة رقم 1 ---
 تم احضار الداتا من API
تم تخزين البيانات في Database
تم إرسال البيانات إلى Webhook
تم تنفيذ الدورة رقم 1 بنجاح


True

In [53]:
import time

for cycle_number in range(1, 4):
    success = run_pipeline(cycle_number)

    if cycle_number < 3:
        print("انتظار 30 ثانية قبل الدورة التالية...")
        time.sleep(30)


--- بدء الدورة رقم 1 ---
 تم احضار الداتا من API
تم تخزين البيانات في Database
تم إرسال البيانات إلى Webhook
تم تنفيذ الدورة رقم 1 بنجاح
انتظار 30 ثانية قبل الدورة التالية...

--- بدء الدورة رقم 2 ---
 تم احضار الداتا من API
تم تخزين البيانات في Database
تم إرسال البيانات إلى Webhook
تم تنفيذ الدورة رقم 2 بنجاح
انتظار 30 ثانية قبل الدورة التالية...

--- بدء الدورة رقم 3 ---
 تم احضار الداتا من API
تم تخزين البيانات في Database
تم إرسال البيانات إلى Webhook
تم تنفيذ الدورة رقم 3 بنجاح


### `اخر تحقق بشوف بيه لو ال 3 دورات اتخزنوا فعلا`

In [54]:
connection = sqlite3.connect("store.db")
cursor = connection.cursor()

cursor.execute("""
    SELECT COUNT(*)
    FROM api_logs
""")

count = cursor.fetchone()[0]

connection.close()

print("Total API records stored:", count)

Total API records stored: 12


### `المف النهائى`

In [55]:
%%writefile W2D4_Automation_Pipeline.py

import requests
import sqlite3
import json
import time
from datetime import datetime


# =========================
# Configuration
# =========================

API_KEY = "YOUR_OPENWEATHER_API_KEY"

API_URL = "https://api.openweathermap.org/data/2.5/weather"

WEBHOOK_URL = "https://kilowatt-yoga-sixties.ngrok-free.dev/webhook"

params = {
    "q": "Alexandria,EG",
    "appid": API_KEY,
    "units": "metric"
}


# =========================
# Database Setup
# =========================

def init_database():
    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS api_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            api_data TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)

    connection.commit()
    connection.close()


# =========================
# Automation Pipeline
# =========================

def run_pipeline(cycle_number):

    print(f"\n--- بدء الدورة رقم {cycle_number} ---")

    # 1. Get data from API
    response = requests.get(
        API_URL,
        params=params,
        timeout=10
    )

    if response.status_code != 200:
        print("فشل الاتصال بالـ API")
        print("Status Code:", response.status_code)
        print("Response:", response.text)
        return False

    data = response.json()

    print("تم جلب البيانات من API")

    # 2. Store data in Database
    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute(
        """
        INSERT INTO api_logs (api_data, timestamp)
        VALUES (?, ?)
        """,
        (
            json.dumps(data, ensure_ascii=False),
            datetime.now().isoformat()
        )
    )

    connection.commit()
    connection.close()

    print("تم تخزين البيانات في Database")

    # 3. Send the same data to Webhook
    webhook_response = requests.post(
        WEBHOOK_URL,
        json=data,
        timeout=10
    )

    if webhook_response.status_code != 200:
        print("فشل إرسال البيانات إلى Webhook")
        print("Status Code:", webhook_response.status_code)
        print("Response:", webhook_response.text)
        return False

    print("تم إرسال البيانات إلى Webhook")

    print(f"تم تنفيذ الدورة رقم {cycle_number} بنجاح")

    return True


# =========================
# Main Automation
# =========================

if __name__ == "__main__":

    init_database()

    for cycle_number in range(1, 4):

        success = run_pipeline(cycle_number)

        if success and cycle_number < 3:
            print("انتظار 30 ثانية قبل الدورة التالية...")
            time.sleep(30)

Overwriting W2D4_Automation_Pipeline.py


### `بتأكد ان الملف موجود`

In [56]:
import os

print(os.path.exists("W2D4_Automation_Pipeline.py"))

True


In [57]:
with open("W2D4_Automation_Pipeline.py", "r", encoding="utf-8") as file:
    content = file.read()

print("YOUR_OPENWEATHER_API_KEY" in content)

True


In [58]:
with open("W2D4_Automation_Pipeline.py", "r", encoding="utf-8") as file:
    print(file.read())


import requests
import sqlite3
import json
import time
from datetime import datetime


# =========================
# Configuration
# =========================

API_KEY = "YOUR_OPENWEATHER_API_KEY"

API_URL = "https://api.openweathermap.org/data/2.5/weather"

WEBHOOK_URL = "https://kilowatt-yoga-sixties.ngrok-free.dev/webhook"

params = {
    "q": "Alexandria,EG",
    "appid": API_KEY,
    "units": "metric"
}


# =========================
# Database Setup
# =========================

def init_database():
    connection = sqlite3.connect("store.db")
    cursor = connection.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS api_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            api_data TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)

    connection.commit()
    connection.close()


# =========================
# Automation Pipeline
# =========================

def run_pipeline(cycle_number):

    print(f"\n--- بدء الدورة رقم {cycle

In [59]:
from google.colab import userdata

API_KEY = userdata.get("OPENWEATHER_API_KEY")

print("API Key loaded:", API_KEY is not None)

API Key loaded: True


In [60]:
from google.colab import userdata

API_KEY = userdata.get("OPENWEATHER_API_KEY")

print("API Key loaded:", API_KEY is not None)
print("API Key length:", len(API_KEY) if API_KEY else 0)

API Key loaded: True
API Key length: 33


In [61]:
API_URL = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": "Alexandria,EG",
    "appid": API_KEY,
    "units": "metric"
}

print("Parameters updated")

Parameters updated
